# Build cache CGHNet (lưới 128×128×16) — notebook CHỈ để build

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

⚠️ **Accelerator = None.** Đây là SimpleITK + numpy, GPU **không** nhanh hơn mà đốt quota
30h/tuần. Notebook này không train gì, không dựng model nào.

## Vì sao có cache này: z = 14–16 là biến lớn nhất dự án chưa thử

Mọi thí nghiệm của dự án đều z = 32 hoặc 48. Cả baseline official của challenge lẫn CGHNet
đều **z = 16**. Ta đang nội suy gấp đôi số lát so với toàn bộ văn liệu, trong khi WORKLOG
S-029 đã ghi nhận khoảng một nửa dataset **bị nội suy vượt** độ phân giải máy chụp ghi được.

Bằng chứng định lượng, CGHNet Bảng 1, tất cả trên đúng test-104 official, cùng protocol
16×128×128 → crop 14×112×112:

| | macro-F1 |
|---|---|
| **ResNet3D trần** | **0.709** |
| ta (E4, trung bình 5 model đơn) | **0.6001** |

**Chênh 0.109 trên cùng một họ kiến trúc.** Đó không phải kiến trúc, đó là protocol.

## Hình học

Nguồn: Li và cs., *Comput Med Imaging Graph* **132** (2026) 102780, §4.3 nguyên văn:
*"all lesion volumes were spatially normalized to a fixed size of 16 × 128 × 128 via
trilinear interpolation ... Random crops of size 14 × 112 × 112 ... deterministic center
cropping was used during inference."*

Thứ tự của họ là D×H×W, nên trong quy ước `[X, Y, Z]` của dự án:

| | giá trị |
|---|---|
| lưới cache | **128 × 128 × 16** |
| model nhận | **112 × 112 × 14** |
| lề mỗi phía | 8 trong mặt phẳng, 1 theo z |

## ⚠️ Cache này KHÔNG dùng được cho config DenseNet

DenseNet121-3D hạ mẫu 5 lần và cần ≥ 32 mọi chiều; z=14 sẽ sập giữa transition layer với
thông báo không nói gì về nguyên nhân (WORKLOG S-063). Cache này **chỉ** dùng cho
`configs/cghnet.yaml` (ViT + ResNet-3D). `tests/test_models.py` có cổng chặn cả hai chiều.

## 0. Bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

PREPROCESS_NAME = "preprocess_cghnet.yaml"
CACHE_DIR = Path("/kaggle/working/cache_cghnet")

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

# build_cache cần SimpleITK. KHÔNG cần torch/monai — notebook này không dựng model.
try:
    import SimpleITK  # noqa: F401
    print("SimpleITK: đã có")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "SimpleITK==2.4.0"], check=True)
    print("SimpleITK: vừa cài")

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

PRE = load_yaml(REPO / "configs" / PREPROCESS_NAME)
INNER = tuple(PRE["target_size"])
MARGIN = tuple(PRE["crop_margin_voxels"])
GRID = tuple(s + 2 * m for s, m in zip(INNER, MARGIN))

assert GRID == (128, 128, 16), f"lưới {GRID}, bài dùng 128x128x16"
print(f"\nconfig:     {PREPROCESS_NAME}")
print(f"lưới cache: {GRID}   (target_size {INNER} + lề {MARGIN} mỗi bên)")
print(f"ghi vào:    {CACHE_DIR}")

## 1. Dữ liệu gốc

`resolve_data_root` trả về `config['data_root']` **mà không xác minh** khi mọi cách dò đều
trượt. Trên Kaggle nó sẽ là một đường dẫn tương đối không tồn tại, và job sẽ chết giữa
chừng. Cell này xác minh trước.

In [ ]:
from src.utils.io import resolve_data_root

cfg_data = load_yaml(REPO / "configs" / "data.yaml")
try:
    data_root = resolve_data_root(cfg_data)
except Exception as exc:  # noqa: BLE001 - chỉ để báo cáo
    data_root, exc_msg = None, str(exc)
else:
    exc_msg = None

ann = (data_root / cfg_data["annotation_rel"]) if data_root else None
if ann is None or not ann.exists():
    raise RuntimeError(
        f"Không tìm thấy dữ liệu LLD-MMRI gốc.\n"
        f"  resolve_data_root -> {data_root}"
        + (f" (lỗi: {exc_msg})" if exc_msg else f", nhưng {ann} không tồn tại")
        + f"\n  Cần mount dataset chứa {cfg_data['annotation_rel']}.\n"
        f"  Ứng viên khai trong configs/data.yaml: {cfg_data.get('data_root_candidates')}"
    )
print("data root:", data_root, "✓")
print("annotation:", ann.name, "✓")

## 2. Build (~20 phút)

Nhỏ hơn cache E4: 262k voxel/ca so với 401k, nên nhanh hơn và chỉ ~2,0 GB (E4 3,2 GB,
E12 5,9 GB).

**Resume được**: bệnh nhân đã có `.npz` thì bỏ qua. Session bị ngắt thì chạy lại đúng cell
này, nó đi tiếp từ chỗ dừng.

In [ ]:
import time

t0 = time.time()
rc = subprocess.run(
    [sys.executable, "-m", "src.preprocess.build_cache", "--config", f"configs/{PREPROCESS_NAME}"],
    cwd=REPO,
).returncode
assert rc == 0, "build cache thất bại — đọc log ở trên"
print(f"\nbuild xong sau {(time.time() - t0) / 60:.0f} phút")

## Cổng nghiệm thu ⚠️

Chạy ở đây chứ không ở notebook train: phát hiện cache hỏng sau khi đã upload 2 GB là quá
muộn. Bốn thứ, mỗi thứ chặn một lỗi khác.

In [ ]:
import json as _json

import numpy as np

from src.data.transforms import CenterCrop3D

meta = _json.loads((CACHE_DIR / "cache_meta.json").read_text("utf-8"))
CAN = {
    "align_phases": "per_phase",
    "crop_mode": "lesion_tight",
    "target_size": list(INNER),
    "crop_margin_voxels": list(MARGIN),
}
for k, v in CAN.items():
    assert meta.get(k) == v, f"cache SAI: {k} = {meta.get(k)!r}, cần {v!r}"
assert meta["lesion_tight"]["source"] == "mask", "phải cắt theo mask, không phải bbox"

n_npz = len(list(CACHE_DIR.glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz}/498 ca — build chưa xong, chạy lại cell trên"

# Hình dạng mảng THẬT. Đây là chỗ bắt được `crop_margin_voxels` bị bỏ qua: nếu nó không
# có tác dụng thì mọi thứ khác vẫn đúng, chỉ mảng là 112x112x14.
mau = sorted(CACHE_DIR.glob("*.npz"))[0]
with np.load(mau) as z:
    shape = tuple(z["image"].shape)
    assert shape == (8, *GRID), f"mảng {shape}, cần {(8, *GRID)} — lề dư không có tác dụng"
    assert tuple(z["crop_margin_voxels"]) == MARGIN
    assert tuple(z["inner_size"]) == INNER
    img = z["image"].astype(np.float32)

cut = CenterCrop3D(INNER)({"image": img})["image"]
assert tuple(cut.shape) == (8, *INNER), cut.shape

tong_gb = sum(f.stat().st_size for f in CACHE_DIR.glob("*.npz")) / 2**30
print(f"✓ {n_npz} ca · mảng {shape} · cắt giữa -> {tuple(cut.shape)} · {tong_gb:.1f} GB")
print(f"  commit build: {meta.get('git_commit')}")
print("\n⚠ Cache này z=14, KHÔNG dùng được cho config DenseNet121 (cần >= 32 mọi chiều).")

## 3. Lưu thành Kaggle Dataset

⚠️ **Để Private.** LLD-MMRI dùng license CC BY-NC-ND, không được phát tán bản phái sinh.

**Cách dễ nhất:** *Save Version* → *Save & Run All*. Xong thì mount được vào notebook khác
qua *Add Data → Your Work → Notebook Output*.

Cell dưới in lệnh CLI nếu bạn muốn tạo Dataset riêng.

In [ ]:
SLUG = "lldmmri-cache-cghnet"
USER = "marcohoang"          # đổi nếu upload bằng tài khoản khác

(CACHE_DIR / "dataset-metadata.json").write_text(
    _json.dumps({
        "title": "LLD-MMRI cache CGHNet (128x128x16, le 8/8/1, per-phase align)",
        "id": f"{USER}/{SLUG}",
        "licenses": [{"name": "other"}],
    }),
    encoding="utf-8",
)

print("Lần đầu:")
print(f"  !kaggle datasets create -p {CACHE_DIR} --dir-mode zip")
print("Các lần sau:")
print(f"  !kaggle datasets version -p {CACHE_DIR} -m 'rebuild' --dir-mode zip")
print("""
Sau khi có Dataset: mở notebooks/19_cghnet.ipynb, mount cache này vào, KHÔNG cần mount dữ
liệu gốc nữa. Notebook 19 nhận diện cache bằng nội dung cache_meta.json chứ không bằng tên,
nên đặt slug gì cũng được — và nó LOẠI cache E4 lẫn cache E12 bằng target_size + lề.
""")